# 15.3 如何決定找誰處理？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**所有expert都forward，MoE是不是已經少算？

soft router先讓每位廚師都做一份，再按比例混合。容易理解但沒有稀疏計算優勢；用它過渡到top-k。

**把直覺寫成數學：**p=softmax(W_router x)，o=Σ_e p_e expert_e(x)。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
router = nn.Linear(4, 3)
x = torch.ones(2, 4)
p = router(x).softmax(-1)
expert_outputs = torch.stack([x, 2 * x, 3 * x], dim=1)
output = (p[:, :, None] * expert_outputs).sum(1)
print(p, output)
assert torch.allclose(p.sum(-1), torch.ones(2))

**如何讀結果：**三個expert輸出在此手設，方便核對混合；真實FFN需要獨立權重。

**只改一件事：**只把router權重設零，觀察均勻混合。
